# Notebook 03 — Dependency Graph and Automatic Leakage Compiler

## Purpose

This notebook converts the expert-audited battery-property leakage taxonomy from Notebook 02 into a machine-readable, reusable compiler.

It performs **no machine-learning training or DFT calculation**.

### Main outputs

- battery-property dependency graph (`JSON`, `GraphML`, node/edge tables)
- machine-readable leakage and protocol rules (`YAML`)
- automatically compiled target-specific leakage classes
- automatically compiled P0–P4 feature sets
- expert-versus-compiler agreement audits
- reusable `electrode_audit` Python module
- hashes, manifests, and a final go/no-go decision

### Scientific gate

The notebook proceeds only when:

1. Notebook 02 reports a full go decision;
2. all Notebook 02 protocol-integrity checks pass;
3. the compiler reproduces every expert leakage class;
4. no expert-unsafe feature is automatically classified as safe;
5. the compiler reproduces every P0–P4 inclusion decision.


In [ ]:
# Cell 1 — Imports, deterministic settings, and dependency check
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import re
import shutil
import sys

sys.dont_write_bytecode = True
import textwrap
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

REQUIRED_IMPORTS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "networkx": "networkx",
    "yaml": "PyYAML",
}

missing = []
for module_name, package_name in REQUIRED_IMPORTS.items():
    try:
        __import__(module_name)
    except Exception:
        missing.append(package_name)

if missing:
    raise RuntimeError(
        "Missing required packages: " + ", ".join(missing) +
        "\nInstall them in a new Jupyter cell with:\n%pip install pandas numpy networkx pyyaml"
    )

import networkx as nx
import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
POLICY_VERSION = "03_dependency_compiler_v1.0"

print("Run timestamp (UTC):", RUN_TIMESTAMP_UTC)
print("Policy version:", POLICY_VERSION)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("networkx:", nx.__version__)


In [ ]:
# Cell 2 — Clean-room repository configuration and Notebook 02 input resolution
import sys

def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

WORKSPACE_ROOT = REPOSITORY_ROOT
OUTPUT_ROOT = artifact_namespace("03", REPOSITORY_ROOT)
PROCESSED_DIR = OUTPUT_ROOT / "processed"
AUDIT_DIR = OUTPUT_ROOT / "audit"
METADATA_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs"
SOFTWARE_DIR = OUTPUT_ROOT / "software"
PACKAGE_DIR = SOFTWARE_DIR / "electrode_audit"
TEST_DIR = OUTPUT_ROOT / "tests"

for directory in [PROCESSED_DIR, AUDIT_DIR, METADATA_DIR, LOG_DIR, PACKAGE_DIR, TEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_RELATIVE_FILES = [
    Path("audit/02_target_specific_leakage_matrix.csv"),
    Path("audit/02_feature_leakage_taxonomy_master.csv"),
    Path("audit/02_protocol_feature_counts_by_target.csv"),
    Path("audit/02_protocol_integrity_audit.csv"),
    Path("metadata/02_protocol_definitions.json"),
    Path("metadata/02_final_decision.json"),
    Path("processed/02_master_feature_table.csv"),
    Path("processed/02_master_metadata_and_targets.csv"),
]


def is_valid_notebook02_root(namespace) -> bool:
    return all((namespace / rel).is_file() for rel in REQUIRED_RELATIVE_FILES)


INPUT_ROOT = artifact_namespace("02", REPOSITORY_ROOT)
if not is_valid_notebook02_root(INPUT_ROOT):
    missing = [
        str(INPUT_ROOT / rel)
        for rel in REQUIRED_RELATIVE_FILES
        if not (INPUT_ROOT / rel).is_file()
    ]
    raise FileNotFoundError(
        "Canonical Notebook 02 inputs are incomplete:\n" + "\n".join(missing)
    )

INPUT_DISCOVERY_METHOD = "canonical_repository_namespace"

print("Workspace root:", WORKSPACE_ROOT)
print("Notebook 02 input namespace:", INPUT_ROOT)
print("Discovery method:", INPUT_DISCOVERY_METHOD)
print("Notebook 03 output namespace:", OUTPUT_ROOT)


In [ ]:
# Cell 3 — Load and validate Notebook 02 evidence
MATRIX_PATH = INPUT_ROOT / "audit/02_target_specific_leakage_matrix.csv"
TAXONOMY_PATH = INPUT_ROOT / "audit/02_feature_leakage_taxonomy_master.csv"
PROTOCOL_COUNTS_PATH = INPUT_ROOT / "audit/02_protocol_feature_counts_by_target.csv"
INTEGRITY_PATH = INPUT_ROOT / "audit/02_protocol_integrity_audit.csv"
PROTOCOL_DEFINITIONS_PATH = INPUT_ROOT / "metadata/02_protocol_definitions.json"
FINAL_DECISION_PATH = INPUT_ROOT / "metadata/02_final_decision.json"
MASTER_FEATURE_PATH = INPUT_ROOT / "processed/02_master_feature_table.csv"
METADATA_TARGETS_PATH = INPUT_ROOT / "processed/02_master_metadata_and_targets.csv"

expert_matrix = pd.read_csv(MATRIX_PATH)
feature_taxonomy = pd.read_csv(TAXONOMY_PATH)
protocol_counts = pd.read_csv(PROTOCOL_COUNTS_PATH)
protocol_integrity = pd.read_csv(INTEGRITY_PATH)
protocol_definitions = json.loads(PROTOCOL_DEFINITIONS_PATH.read_text(encoding="utf-8"))
notebook09_decision = json.loads(FINAL_DECISION_PATH.read_text(encoding="utf-8"))

required_matrix_columns = {
    "target", "target_unit", "feature", "feature_origin", "leakage_level", "leakage_reason",
    "allowed_P0", "allowed_P1", "allowed_P2", "allowed_P3", "allowed_P4",
}
missing_matrix_columns = sorted(required_matrix_columns - set(expert_matrix.columns))
if missing_matrix_columns:
    raise RuntimeError(f"Notebook 02 leakage matrix is missing columns: {missing_matrix_columns}")

if notebook09_decision.get("final_decision") != "FULL_GO_TO_NOTEBOOK_10":
    raise RuntimeError(
        "Notebook 02 did not report FULL_GO_TO_NOTEBOOK_10. "
        f"Observed: {notebook09_decision.get('final_decision')}"
    )

failed_integrity = protocol_integrity.loc[protocol_integrity["status"].astype(str).str.upper() != "PASS"]
if not failed_integrity.empty:
    raise RuntimeError(
        "Notebook 02 protocol-integrity audit contains non-PASS rows:\n" +
        failed_integrity.to_string(index=False)
    )

if expert_matrix.duplicated(["target", "feature"]).any():
    dup = expert_matrix.loc[expert_matrix.duplicated(["target", "feature"], keep=False)]
    raise RuntimeError("Duplicate target-feature rows detected:\n" + dup.head(20).to_string(index=False))

if not set(expert_matrix["leakage_level"].unique()).issubset({"L0", "L1", "L2", "L3", "L4", "L5"}):
    raise RuntimeError("Unexpected leakage classes found in Notebook 02 matrix.")

TARGETS = protocol_definitions["targets"]
PRIMARY_TARGETS = protocol_definitions["primary_benchmark_targets"]
TARGET_UNITS = protocol_definitions["target_units"]
PROTOCOLS = ["P0", "P1", "P2", "P3", "P4"]

expected_pairs = len(TARGETS) * feature_taxonomy["feature"].nunique()
if len(expert_matrix) != expected_pairs:
    raise RuntimeError(
        f"Expected {expected_pairs} target-feature rows but found {len(expert_matrix)}."
    )

print("Targets:", TARGETS)
print("Primary benchmark targets:", PRIMARY_TARGETS)
print("Unique candidate features:", feature_taxonomy["feature"].nunique())
print("Target-feature audit rows:", len(expert_matrix))
print("Notebook 02 integrity rows:", len(protocol_integrity), "all PASS")


In [ ]:
# Cell 4 — Write the reusable compiler module
MODULE_CODE = '"""Dependency-aware leakage compiler for computed insertion-electrode properties.\n\nThe compiler applies machine-readable, target-specific physical and workflow rules.\nIt does not train predictive models.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict\nfrom typing import Dict, Iterable, List\n\nCAPACITY_TARGETS = {"capacity_grav", "capacity_vol"}\nENERGY_TARGETS = {"energy_grav", "energy_vol"}\nSTABILITY_TARGETS = {"stability_charge", "stability_discharge", "stability_worst"}\nCOMPOSITION_ORIGINS = {"composition_known_framework", "composition_known_working_ion"}\n\n\n@dataclass(frozen=True)\nclass LeakageDecision:\n    target: str\n    feature: str\n    feature_origin: str\n    leakage_level: str\n    leakage_reason: str\n    rule_id: str\n    dependency_path: str\n\n    def to_dict(self) -> Dict[str, str]:\n        return asdict(self)\n\n\ndef classify_feature(feature: str, feature_origin: str, target: str) -> LeakageDecision:\n    """Classify a feature for one target using ordered physical/workflow rules."""\n    if feature_origin == "domain_label_onehot":\n        return LeakageDecision(target, feature, feature_origin, "L5",\n            "Working-ion/domain labels encode validation-domain membership and must be controlled by split design.",\n            "RULE_DOMAIN_LABEL", f"feature::{feature} -> concept::validation_domain -> target::{target}")\n\n    if feature == target:\n        return LeakageDecision(target, feature, feature_origin, "L1",\n            "The feature is the exact target column.",\n            "RULE_DIRECT_TARGET", f"feature::{feature} -> target::{target}")\n\n    if feature_origin == "stored_electrode_target":\n        if target == "average_voltage" and feature in ENERGY_TARGETS:\n            return LeakageDecision(target, feature, feature_origin, "L2",\n                "Stored energy is mathematically coupled to voltage and capacity.",\n                "RULE_ENERGY_VOLTAGE_RELATION",\n                f"feature::{feature} -> concept::energy_definition -> target::{target}")\n        if target in CAPACITY_TARGETS and feature in (CAPACITY_TARGETS | ENERGY_TARGETS):\n            return LeakageDecision(target, feature, feature_origin, "L2",\n                "Capacity and energy variants are mathematically target-adjacent.",\n                "RULE_CAPACITY_ENERGY_RELATION",\n                f"feature::{feature} -> concept::capacity_energy_relation -> target::{target}")\n        if target in ENERGY_TARGETS and feature in ({"average_voltage"} | CAPACITY_TARGETS | ENERGY_TARGETS):\n            return LeakageDecision(target, feature, feature_origin, "L2",\n                "Energy is defined by voltage and capacity, with unit-related energy/capacity variants.",\n                "RULE_ENERGY_DEFINITION",\n                f"feature::{feature} -> concept::energy_definition -> target::{target}")\n        if target in STABILITY_TARGETS and feature in STABILITY_TARGETS:\n            return LeakageDecision(target, feature, feature_origin, "L2",\n                "Endpoint and worst-case stability values are definitionally linked.",\n                "RULE_STABILITY_AGGREGATION",\n                f"feature::{feature} -> concept::stability_aggregation -> target::{target}")\n        return LeakageDecision(target, feature, feature_origin, "L4",\n            "A different stored electrode property is available only as a post-hoc computed-record descriptor.",\n            "RULE_OTHER_STORED_TARGET",\n            f"feature::{feature} -> concept::stored_electrode_record -> target::{target}")\n\n    if feature_origin == "stoichiometric_window":\n        if target in (CAPACITY_TARGETS | ENERGY_TARGETS):\n            return LeakageDecision(target, feature, feature_origin, "L3",\n                "Working-ion insertion stoichiometry directly controls theoretical capacity and energy scale.",\n                "RULE_STOICHIOMETRIC_WINDOW",\n                f"feature::{feature} -> concept::working_ion_transfer -> target::{target}")\n        return LeakageDecision(target, feature, feature_origin, "L0",\n            "No direct target dependency is identified for this target.",\n            "RULE_SAFE_DEFAULT", f"feature::{feature} -> target::{target}")\n\n    if feature_origin == "stored_electrode_process_descriptor":\n        if feature == "max_voltage_step" and target == "average_voltage":\n            return LeakageDecision(target, feature, feature_origin, "L4",\n                "A stored voltage-window descriptor is adjacent to the voltage target.",\n                "RULE_VOLTAGE_WINDOW",\n                f"feature::{feature} -> concept::voltage_window -> target::{target}")\n        return LeakageDecision(target, feature, feature_origin, "L0",\n            "No direct target dependency is identified for this target.",\n            "RULE_SAFE_DEFAULT", f"feature::{feature} -> target::{target}")\n\n    if feature_origin in {"composition_known_framework", "composition_known_working_ion", "other_numeric"}:\n        return LeakageDecision(target, feature, feature_origin, "L0",\n            "No direct target dependency is identified for this target.",\n            "RULE_SAFE_DEFAULT", f"feature::{feature} -> target::{target}")\n\n    if feature_origin == "post_dft_electronic_summary":\n        return LeakageDecision(target, feature, feature_origin, "L4",\n            "The electronic descriptor is generated in the same post-DFT record and is decision-support only.",\n            "RULE_POST_DFT_ELECTRONIC",\n            f"feature::{feature} -> concept::post_dft_record -> target::{target}")\n\n    if feature_origin == "post_dft_provenance_flag":\n        return LeakageDecision(target, feature, feature_origin, "L4",\n            "The provenance flag is available only from the post-DFT record.",\n            "RULE_POST_DFT_PROVENANCE",\n            f"feature::{feature} -> concept::post_dft_record -> target::{target}")\n\n    if feature_origin == "post_dft_energy_stability_summary":\n        if "formation_energy" in feature:\n            if target == "average_voltage":\n                return LeakageDecision(target, feature, feature_origin, "L2",\n                    "Endpoint formation energies can reconstruct insertion-voltage trends.",\n                    "RULE_FORMATION_ENERGY_VOLTAGE",\n                    f"feature::{feature} -> concept::insertion_energy_difference -> target::{target}")\n            if target in ENERGY_TARGETS:\n                return LeakageDecision(target, feature, feature_origin, "L2",\n                    "Endpoint formation energies are target-defining or strongly target-adjacent for electrode energy.",\n                    "RULE_FORMATION_ENERGY_ENERGY",\n                    f"feature::{feature} -> concept::insertion_energy_difference -> target::{target}")\n            if target in STABILITY_TARGETS:\n                return LeakageDecision(target, feature, feature_origin, "L4",\n                    "Formation energy is post-DFT adjacent to stability but is not the direct hull target.",\n                    "RULE_FORMATION_ENERGY_STABILITY",\n                    f"feature::{feature} -> concept::post_dft_thermodynamics -> target::{target}")\n            return LeakageDecision(target, feature, feature_origin, "L4",\n                "The thermodynamic summary is a post-DFT decision-support descriptor.",\n                "RULE_POST_DFT_ENERGY_DEFAULT",\n                f"feature::{feature} -> concept::post_dft_thermodynamics -> target::{target}")\n\n        if ("energy_above_hull" in feature) or ("is_stable" in feature):\n            if target in STABILITY_TARGETS:\n                return LeakageDecision(target, feature, feature_origin, "L2",\n                    "Endpoint hull/stability descriptors define or directly reconstruct stability targets.",\n                    "RULE_HULL_STABILITY",\n                    f"feature::{feature} -> concept::hull_stability -> target::{target}")\n            return LeakageDecision(target, feature, feature_origin, "L4",\n                "The hull/stability summary is a post-DFT decision-support descriptor for this target.",\n                "RULE_HULL_OTHER",\n                f"feature::{feature} -> concept::post_dft_thermodynamics -> target::{target}")\n\n        raise ValueError(f"Unrecognized post-DFT energy/stability feature: {feature}")\n\n    if feature_origin == "relaxed_structure_summary":\n        if target == "max_delta_volume" and any(token in feature for token in ("volume", "density")):\n            return LeakageDecision(target, feature, feature_origin, "L2",\n                "Endpoint volume/density descriptors define or strongly encode the volume-change target.",\n                "RULE_VOLUME_CHANGE",\n                f"feature::{feature} -> concept::volume_change_definition -> target::{target}")\n        return LeakageDecision(target, feature, feature_origin, "L0",\n            "The relaxed-structure summary is not target-defining for this target.",\n            "RULE_SAFE_STRUCTURE", f"feature::{feature} -> target::{target}")\n\n    raise ValueError(\n        f"No compiler rule for feature={feature!r}, origin={feature_origin!r}, target={target!r}"\n    )\n\n\ndef protocol_allows(protocol: str, leakage_level: str, feature_origin: str) -> bool:\n    """Apply deployment-stage and leakage-level policy for P0-P4."""\n    if protocol == "P0":\n        return leakage_level in {"L0", "L2", "L3", "L4"}\n    if protocol == "P1":\n        return leakage_level == "L0" and feature_origin in COMPOSITION_ORIGINS\n    if protocol == "P2":\n        return leakage_level == "L0" and feature_origin in (COMPOSITION_ORIGINS | {"relaxed_structure_summary"})\n    if protocol == "P3":\n        if feature_origin in COMPOSITION_ORIGINS:\n            return leakage_level == "L0"\n        if feature_origin == "relaxed_structure_summary":\n            return leakage_level == "L0"\n        if feature_origin == "stored_electrode_process_descriptor":\n            return leakage_level in {"L0", "L4"}\n        if feature_origin in {"post_dft_electronic_summary", "post_dft_energy_stability_summary"}:\n            return leakage_level == "L4"\n        return False\n    if protocol == "P4":\n        return leakage_level in {"L0", "L1", "L2", "L3", "L4"}\n    raise ValueError(f"Unknown protocol: {protocol}")\n\n\ndef compile_target_feature_rows(feature_records: Iterable[Dict[str, str]], targets: Iterable[str]) -> List[Dict[str, object]]:\n    rows: List[Dict[str, object]] = []\n    for target in targets:\n        for record in feature_records:\n            decision = classify_feature(record["feature"], record["feature_origin"], target)\n            row: Dict[str, object] = decision.to_dict()\n            for protocol in ("P0", "P1", "P2", "P3", "P4"):\n                row[f"allowed_{protocol}"] = protocol_allows(protocol, decision.leakage_level, decision.feature_origin)\n            rows.append(row)\n    return rows\n'

(PACKAGE_DIR / "__init__.py").write_text(
    '"""Reusable dependency-aware leakage tools for insertion-electrode datasets."""\n'
    'from .leakage_compiler import LeakageDecision, classify_feature, protocol_allows, compile_target_feature_rows\n',
    encoding="utf-8",
)
(PACKAGE_DIR / "leakage_compiler.py").write_text(MODULE_CODE, encoding="utf-8")

# Import the newly written module without modifying the global environment.
spec = importlib.util.spec_from_file_location(
    "electrode_audit.leakage_compiler", PACKAGE_DIR / "leakage_compiler.py"
)
compiler_module = importlib.util.module_from_spec(spec)
assert spec and spec.loader
sys.modules[spec.name] = compiler_module
spec.loader.exec_module(compiler_module)

print("Reusable compiler module written to:", PACKAGE_DIR / "leakage_compiler.py")


In [ ]:
# Cell 5 — Define machine-readable target, workflow, and protocol rules
ORIGIN_TO_STAGE = {
    "composition_known_framework": "pre_dft_composition",
    "composition_known_working_ion": "pre_dft_composition",
    "stoichiometric_window": "electrode_state_stoichiometry",
    "relaxed_structure_summary": "relaxed_structure",
    "post_dft_energy_stability_summary": "post_dft_thermodynamics",
    "post_dft_electronic_summary": "post_dft_electronic",
    "post_dft_provenance_flag": "post_dft_provenance",
    "stored_electrode_process_descriptor": "stored_electrode_record",
    "stored_electrode_target": "stored_electrode_record",
    "domain_label_onehot": "validation_only",
    "other_numeric": "derived_miscellaneous",
}

RULE_CONFIG = {
    "schema_version": "1.0",
    "policy_version": POLICY_VERSION,
    "targets": TARGETS,
    "primary_benchmark_targets": PRIMARY_TARGETS,
    "target_units": TARGET_UNITS,
    "target_groups": {
        "capacity": ["capacity_grav", "capacity_vol"],
        "energy": ["energy_grav", "energy_vol"],
        "stability": ["stability_charge", "stability_discharge", "stability_worst"],
        "voltage": ["average_voltage"],
        "volume_change": ["max_delta_volume"],
    },
    "physical_dependencies": [
        {"source": "average_voltage", "target": "energy_grav", "relation": "multiplicative_component", "class": "L2"},
        {"source": "capacity_grav", "target": "energy_grav", "relation": "multiplicative_component", "class": "L2"},
        {"source": "average_voltage", "target": "energy_vol", "relation": "multiplicative_component", "class": "L2"},
        {"source": "capacity_vol", "target": "energy_vol", "relation": "multiplicative_component", "class": "L2"},
        {"source": "stability_charge", "target": "stability_worst", "relation": "maximum_component", "class": "L2"},
        {"source": "stability_discharge", "target": "stability_worst", "relation": "maximum_component", "class": "L2"},
        {"source": "working_ion_transfer", "target": "capacity_grav", "relation": "stoichiometric_scale", "class": "L3"},
        {"source": "working_ion_transfer", "target": "capacity_vol", "relation": "stoichiometric_scale", "class": "L3"},
    ],
    "feature_origin_workflow_stage": ORIGIN_TO_STAGE,
    "protocols": {
        "P0": {"name": "full_feature_leaky_baseline_no_direct_target", "deployment_stage": "diagnostic"},
        "P1": {"name": "composition_only_clean", "deployment_stage": "pre_dft"},
        "P2": {"name": "composition_plus_relaxed_structure_clean_strict", "deployment_stage": "post_structure_pre_electrode_property"},
        "P3": {"name": "post_dft_decision_support_clean_target_specific", "deployment_stage": "post_dft_decision_support"},
        "P4": {"name": "leakage_stress_test", "deployment_stage": "diagnostic"},
    },
    "rule_priority": [
        "RULE_DOMAIN_LABEL",
        "RULE_DIRECT_TARGET",
        "RULE_ENERGY_VOLTAGE_RELATION",
        "RULE_CAPACITY_ENERGY_RELATION",
        "RULE_ENERGY_DEFINITION",
        "RULE_STABILITY_AGGREGATION",
        "RULE_STOICHIOMETRIC_WINDOW",
        "RULE_FORMATION_ENERGY_VOLTAGE",
        "RULE_FORMATION_ENERGY_ENERGY",
        "RULE_HULL_STABILITY",
        "RULE_VOLUME_CHANGE",
        "RULE_POST_DFT_*",
        "RULE_SAFE_*",
    ],
}

RULES_YAML_PATH = PROCESSED_DIR / "03_leakage_rules.yaml"
RULES_YAML_PATH.write_text(
    yaml.safe_dump(RULE_CONFIG, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)
print("Machine-readable rules written to:", RULES_YAML_PATH)


In [ ]:
# Cell 6 — Compile leakage classes and P0–P4 feature eligibility
feature_registry = (
    expert_matrix[["feature", "feature_origin"]]
    .drop_duplicates()
    .sort_values(["feature_origin", "feature"])
    .reset_index(drop=True)
)

unknown_origins = sorted(set(feature_registry["feature_origin"]) - set(ORIGIN_TO_STAGE))
if unknown_origins:
    raise RuntimeError(f"Feature origins without workflow-stage mapping: {unknown_origins}")

compiled_rows = compiler_module.compile_target_feature_rows(
    feature_registry.to_dict(orient="records"), TARGETS
)
compiled = pd.DataFrame(compiled_rows)
compiled["target_unit"] = compiled["target"].map(TARGET_UNITS)
compiled["workflow_stage"] = compiled["feature_origin"].map(ORIGIN_TO_STAGE)
compiled["is_primary_benchmark_target"] = compiled["target"].isin(PRIMARY_TARGETS)

compiled = compiled[[
    "target", "target_unit", "is_primary_benchmark_target",
    "feature", "feature_origin", "workflow_stage",
    "leakage_level", "leakage_reason", "rule_id", "dependency_path",
    "allowed_P0", "allowed_P1", "allowed_P2", "allowed_P3", "allowed_P4",
]].sort_values(["target", "feature"]).reset_index(drop=True)

if len(compiled) != len(expert_matrix):
    raise RuntimeError(f"Compiler produced {len(compiled)} rows; expected {len(expert_matrix)}.")

COMPILED_MATRIX_PATH = PROCESSED_DIR / "03_compiled_target_feature_matrix.csv"
compiled.to_csv(COMPILED_MATRIX_PATH, index=False)

# Long feature-set table for direct consumption by later notebooks.
feature_set_rows = []
for target in TARGETS:
    target_rows = compiled.loc[compiled["target"] == target]
    for protocol in PROTOCOLS:
        selected = target_rows.loc[target_rows[f"allowed_{protocol}"]].copy()
        for rank, row in enumerate(selected.itertuples(index=False), start=1):
            feature_set_rows.append({
                "target": target,
                "target_unit": TARGET_UNITS[target],
                "protocol": protocol,
                "feature_order": rank,
                "feature": row.feature,
                "feature_origin": row.feature_origin,
                "workflow_stage": row.workflow_stage,
                "compiler_leakage_level": row.leakage_level,
                "compiler_rule_id": row.rule_id,
            })

compiled_feature_sets = pd.DataFrame(feature_set_rows)
COMPILED_FEATURE_SETS_PATH = PROCESSED_DIR / "03_compiled_feature_sets_by_target.csv"
compiled_feature_sets.to_csv(COMPILED_FEATURE_SETS_PATH, index=False)

excluded = compiled.loc[
    ~compiled[[f"allowed_{p}" for p in PROTOCOLS]].all(axis=1)
].copy()
EXCLUSION_PATH = PROCESSED_DIR / "03_compiled_exclusion_reasons.csv"
excluded.to_csv(EXCLUSION_PATH, index=False)

print("Compiled target-feature rows:", len(compiled))
print("Compiled protocol-feature rows:", len(compiled_feature_sets))
print("Compiled matrix:", COMPILED_MATRIX_PATH)


In [ ]:
# Cell 7 — Validate compiler against the expert-audited Notebook 02 matrix
manual_columns = [
    "target", "feature", "feature_origin", "leakage_level", "leakage_reason",
    "allowed_P0", "allowed_P1", "allowed_P2", "allowed_P3", "allowed_P4",
]
manual = expert_matrix[manual_columns].copy()
manual = manual.rename(columns={
    "leakage_level": "expert_leakage_level",
    "leakage_reason": "expert_leakage_reason",
    **{f"allowed_{p}": f"expert_allowed_{p}" for p in PROTOCOLS},
})

auto_columns = [
    "target", "feature", "feature_origin", "workflow_stage",
    "leakage_level", "leakage_reason", "rule_id", "dependency_path",
    *[f"allowed_{p}" for p in PROTOCOLS],
]
auto = compiled[auto_columns].copy().rename(columns={
    "leakage_level": "compiler_leakage_level",
    "leakage_reason": "compiler_leakage_reason",
    **{f"allowed_{p}": f"compiler_allowed_{p}" for p in PROTOCOLS},
})

agreement = manual.merge(auto, on=["target", "feature", "feature_origin"], how="outer", indicator=True)
agreement["class_agreement"] = (
    agreement["expert_leakage_level"] == agreement["compiler_leakage_level"]
)
for protocol in PROTOCOLS:
    agreement[f"{protocol}_agreement"] = (
        agreement[f"expert_allowed_{protocol}"].astype("boolean") ==
        agreement[f"compiler_allowed_{protocol}"].astype("boolean")
    )

agreement["all_protocols_agree"] = agreement[[f"{p}_agreement" for p in PROTOCOLS]].all(axis=1)
agreement["false_safe"] = (
    agreement["expert_leakage_level"].isin(["L1", "L2", "L3", "L4", "L5"]) &
    (agreement["compiler_leakage_level"] == "L0")
)
agreement["compiler_more_conservative"] = (
    (agreement["expert_leakage_level"] == "L0") &
    agreement["compiler_leakage_level"].isin(["L1", "L2", "L3", "L4", "L5"])
)

AGREEMENT_PATH = AUDIT_DIR / "03_manual_vs_compiler_agreement.csv"
agreement.to_csv(AGREEMENT_PATH, index=False)

summary_rows = []
for target in ["ALL", *TARGETS]:
    subset = agreement if target == "ALL" else agreement.loc[agreement["target"] == target]
    summary_rows.append({
        "target": target,
        "n_rows": len(subset),
        "class_agreement_n": int(subset["class_agreement"].sum()),
        "class_agreement_rate": float(subset["class_agreement"].mean()),
        "false_safe_n": int(subset["false_safe"].sum()),
        "compiler_more_conservative_n": int(subset["compiler_more_conservative"].sum()),
        "all_protocols_agree_n": int(subset["all_protocols_agree"].sum()),
        "all_protocols_agree_rate": float(subset["all_protocols_agree"].mean()),
        **{
            f"{p}_agreement_rate": float(subset[f"{p}_agreement"].mean())
            for p in PROTOCOLS
        },
    })

validation_summary = pd.DataFrame(summary_rows)
VALIDATION_SUMMARY_PATH = AUDIT_DIR / "03_compiler_validation_summary.csv"
validation_summary.to_csv(VALIDATION_SUMMARY_PATH, index=False)

print(validation_summary.to_string(index=False))


In [ ]:
# Cell 8 — Reconstruct protocol counts and compare with Notebook 02
reconstructed_counts = []
for target in TARGETS:
    target_rows = compiled.loc[compiled["target"] == target]
    for protocol in PROTOCOLS:
        selected = target_rows.loc[target_rows[f"allowed_{protocol}"]]
        row = {
            "target": target,
            "protocol": protocol,
            "compiler_n_features": len(selected),
        }
        for level in ["L0", "L1", "L2", "L3", "L4", "L5"]:
            row[f"compiler_n_{level}"] = int((selected["leakage_level"] == level).sum())
        reconstructed_counts.append(row)

reconstructed_counts = pd.DataFrame(reconstructed_counts)
expert_count_columns = ["target", "protocol", "n_features", "n_L0", "n_L1", "n_L2", "n_L3", "n_L4", "n_L5"]
count_audit = protocol_counts[expert_count_columns].merge(
    reconstructed_counts, on=["target", "protocol"], how="outer", indicator=True
)
count_audit["feature_count_agreement"] = count_audit["n_features"] == count_audit["compiler_n_features"]
for level in ["L0", "L1", "L2", "L3", "L4", "L5"]:
    count_audit[f"{level}_count_agreement"] = (
        count_audit[f"n_{level}"] == count_audit[f"compiler_n_{level}"]
    )
count_audit["all_count_fields_agree"] = count_audit[[
    "feature_count_agreement", *[f"{level}_count_agreement" for level in ["L0", "L1", "L2", "L3", "L4", "L5"]]
]].all(axis=1)

COUNT_AUDIT_PATH = AUDIT_DIR / "03_protocol_reconstruction_audit.csv"
count_audit.to_csv(COUNT_AUDIT_PATH, index=False)

print("Protocol rows:", len(count_audit))
print("Rows with complete count agreement:", int(count_audit["all_count_fields_agree"].sum()))
if not count_audit["all_count_fields_agree"].all():
    display(count_audit.loc[~count_audit["all_count_fields_agree"]])


In [ ]:
# Cell 9 — Build the machine-readable dependency graph
G = nx.MultiDiGraph()

# Concept nodes make physical and workflow relationships explicit.
concepts = {
    "energy_definition": "Energy is linked to voltage and capacity.",
    "capacity_energy_relation": "Capacity and energy variants are mathematically coupled.",
    "stability_aggregation": "Worst-case stability is derived from endpoint stability values.",
    "working_ion_transfer": "Transferred working-ion stoichiometry controls theoretical capacity scale.",
    "insertion_energy_difference": "Endpoint energy differences define insertion voltage/energy trends.",
    "hull_stability": "Endpoint hull quantities define stability targets.",
    "volume_change_definition": "Endpoint volume/density changes define volume-change quantities.",
    "post_dft_record": "Descriptor is generated from the same post-DFT record.",
    "post_dft_thermodynamics": "Thermodynamic summary is available after DFT.",
    "stored_electrode_record": "Stored electrode-property context is post hoc.",
    "validation_domain": "Domain membership must be controlled by split design.",
    "voltage_window": "Stored voltage-window descriptor is adjacent to average voltage.",
}
for concept, description in concepts.items():
    G.add_node(f"concept::{concept}", node_type="concept", label=concept, description=description)

for target in TARGETS:
    G.add_node(
        f"target::{target}", node_type="target", label=target,
        unit=str(TARGET_UNITS[target]), primary_benchmark=str(target in PRIMARY_TARGETS),
    )

for row in feature_registry.itertuples(index=False):
    G.add_node(
        f"feature::{row.feature}", node_type="feature", label=row.feature,
        feature_origin=row.feature_origin,
        workflow_stage=ORIGIN_TO_STAGE[row.feature_origin],
    )

for row in compiled.itertuples(index=False):
    G.add_edge(
        f"feature::{row.feature}", f"target::{row.target}",
        edge_type="compiled_leakage_dependency",
        leakage_level=row.leakage_level,
        rule_id=row.rule_id,
        reason=row.leakage_reason,
        dependency_path=row.dependency_path,
        allowed_P0=str(bool(row.allowed_P0)),
        allowed_P1=str(bool(row.allowed_P1)),
        allowed_P2=str(bool(row.allowed_P2)),
        allowed_P3=str(bool(row.allowed_P3)),
        allowed_P4=str(bool(row.allowed_P4)),
    )

# Canonical physical relationships among linked targets.
for dep in RULE_CONFIG["physical_dependencies"]:
    source = dep["source"]
    source_id = f"target::{source}" if source in TARGETS else f"concept::{source}"
    target_id = f"target::{dep['target']}"
    if source_id not in G:
        G.add_node(source_id, node_type="concept", label=source)
    G.add_edge(
        source_id, target_id,
        edge_type="canonical_physical_dependency",
        leakage_level=dep["class"],
        relation=dep["relation"],
    )

nodes_df = pd.DataFrame([
    {"node_id": node, **attrs} for node, attrs in G.nodes(data=True)
])
edges_df = pd.DataFrame([
    {"source": u, "target": v, "edge_key": k, **attrs}
    for u, v, k, attrs in G.edges(keys=True, data=True)
])

NODES_PATH = PROCESSED_DIR / "03_dependency_graph_nodes.csv"
EDGES_PATH = PROCESSED_DIR / "03_dependency_graph_edges.csv"
GRAPHML_PATH = PROCESSED_DIR / "03_battery_property_dependency_graph.graphml"
GRAPH_JSON_PATH = PROCESSED_DIR / "03_battery_property_dependency_graph.json"

nodes_df.to_csv(NODES_PATH, index=False)
edges_df.to_csv(EDGES_PATH, index=False)
nx.write_graphml(G, GRAPHML_PATH)

node_link = nx.node_link_data(G, edges="edges")
GRAPH_JSON_PATH.write_text(json.dumps(node_link, indent=2, default=str), encoding="utf-8")

PATH_AUDIT_PATH = AUDIT_DIR / "03_dependency_path_audit.csv"
compiled[[
    "target", "feature", "feature_origin", "workflow_stage",
    "leakage_level", "rule_id", "dependency_path",
]].to_csv(PATH_AUDIT_PATH, index=False)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())
print("GraphML:", GRAPHML_PATH)


In [ ]:
# Cell 10 — Create API example and reusable test file
api_example = {
    "target": "energy_grav",
    "deployment_protocol": "P2",
    "example_features": [
        "average_voltage",
        "capacity_grav",
        "fw_X_mean",
        "charge_summary_volume",
        "charge_summary_formation_energy_per_atom",
        "wi_is_Na",
    ],
    "compiled_decisions": [],
}
origin_lookup = dict(zip(feature_registry["feature"], feature_registry["feature_origin"]))
for feature in api_example["example_features"]:
    decision = compiler_module.classify_feature(feature, origin_lookup[feature], api_example["target"])
    decision_dict = decision.to_dict()
    decision_dict["allowed"] = compiler_module.protocol_allows(
        api_example["deployment_protocol"], decision.leakage_level, decision.feature_origin
    )
    api_example["compiled_decisions"].append(decision_dict)

API_EXAMPLE_PATH = PROCESSED_DIR / "03_compiler_api_example.json"
API_EXAMPLE_PATH.write_text(json.dumps(api_example, indent=2), encoding="utf-8")

TEST_CODE = """from electrode_audit.leakage_compiler import classify_feature, protocol_allows\n\n\ndef test_energy_definition_is_L2():\n    d = classify_feature(\"average_voltage\", \"stored_electrode_target\", \"energy_grav\")\n    assert d.leakage_level == \"L2\"\n\n\ndef test_stoichiometry_is_L3_for_capacity():\n    d = classify_feature(\"fracA_discharge\", \"stoichiometric_window\", \"capacity_grav\")\n    assert d.leakage_level == \"L3\"\n\n\ndef test_hull_is_L2_for_worst_stability():\n    d = classify_feature(\"summary_worst_energy_above_hull\", \"post_dft_energy_stability_summary\", \"stability_worst\")\n    assert d.leakage_level == \"L2\"\n\n\ndef test_P2_excludes_volume_proxy_for_volume_change():\n    d = classify_feature(\"charge_summary_volume\", \"relaxed_structure_summary\", \"max_delta_volume\")\n    assert d.leakage_level == \"L2\"\n    assert not protocol_allows(\"P2\", d.leakage_level, d.feature_origin)\n"""
(TEST_DIR / "test_leakage_compiler.py").write_text(TEST_CODE, encoding="utf-8")

print(json.dumps(api_example, indent=2))


In [ ]:
# Cell 11 — Final scientific and software go/no-go gates
all_row_keys_present = bool((agreement["_merge"] == "both").all())
class_agreement_complete = bool(agreement["class_agreement"].all())
zero_false_safe = int(agreement["false_safe"].sum()) == 0
protocol_agreement_complete = bool(agreement["all_protocols_agree"].all())
count_reconstruction_complete = bool(count_audit["all_count_fields_agree"].all())
module_exists = (PACKAGE_DIR / "leakage_compiler.py").is_file()
graph_outputs_exist = all(path.is_file() for path in [GRAPHML_PATH, GRAPH_JSON_PATH, NODES_PATH, EDGES_PATH])

# Direct functional assertions for the defining dependencies.
critical_assertions = [
    compiler_module.classify_feature("average_voltage", "stored_electrode_target", "energy_grav").leakage_level == "L2",
    compiler_module.classify_feature("capacity_grav", "stored_electrode_target", "energy_grav").leakage_level == "L2",
    compiler_module.classify_feature("fracA_charge", "stoichiometric_window", "capacity_grav").leakage_level == "L3",
    compiler_module.classify_feature("stability_charge", "stored_electrode_target", "stability_worst").leakage_level == "L2",
    compiler_module.classify_feature("charge_summary_volume", "relaxed_structure_summary", "max_delta_volume").leakage_level == "L2",
    compiler_module.classify_feature("wi_is_Na", "domain_label_onehot", "average_voltage").leakage_level == "L5",
]
critical_dependencies_pass = all(critical_assertions)

GATES = {
    "notebook09_full_go": notebook09_decision.get("final_decision") == "FULL_GO_TO_NOTEBOOK_10",
    "notebook09_integrity_all_pass": failed_integrity.empty,
    "all_target_feature_keys_present": all_row_keys_present,
    "compiler_class_agreement_complete": class_agreement_complete,
    "zero_false_safe_features": zero_false_safe,
    "compiler_protocol_agreement_complete": protocol_agreement_complete,
    "protocol_count_reconstruction_complete": count_reconstruction_complete,
    "critical_dependency_assertions_pass": critical_dependencies_pass,
    "reusable_module_written": module_exists,
    "dependency_graph_outputs_written": graph_outputs_exist,
}

if all(GATES.values()):
    FINAL_DECISION = "FULL_GO_TO_NOTEBOOK_10B"
else:
    FINAL_DECISION = "HOLD_COMPILER_VALIDATION_FAILURE"

GATE_TABLE = pd.DataFrame([
    {"gate": gate, "passed": bool(passed)} for gate, passed in GATES.items()
])
GATE_TABLE_PATH = AUDIT_DIR / "03_go_no_go_gate_audit.csv"
GATE_TABLE.to_csv(GATE_TABLE_PATH, index=False)

print(GATE_TABLE.to_string(index=False))
print("\nFINAL DECISION:", FINAL_DECISION)

if FINAL_DECISION != "FULL_GO_TO_NOTEBOOK_10B":
    failed = [gate for gate, passed in GATES.items() if not passed]
    raise RuntimeError("Notebook 03 failed the following gates: " + ", ".join(failed))


In [ ]:
# Cell 12 — Write manifests, hashes, environment, event log, and final decision

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

input_paths = [
    MATRIX_PATH, TAXONOMY_PATH, PROTOCOL_COUNTS_PATH, INTEGRITY_PATH,
    PROTOCOL_DEFINITIONS_PATH, FINAL_DECISION_PATH, MASTER_FEATURE_PATH, METADATA_TARGETS_PATH,
]
input_hashes = pd.DataFrame([
    {
        "file_name": path.name,
        "relative_path": str(path.relative_to(INPUT_ROOT)).replace("\\", "/"),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "role": "notebook_02_authoritative_input",
    }
    for path in input_paths
])
INPUT_HASH_PATH = METADATA_DIR / "03_input_file_hashes.csv"
input_hashes.to_csv(INPUT_HASH_PATH, index=False)

environment = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "policy_version": POLICY_VERSION,
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "networkx": nx.__version__,
    "pyyaml": yaml.__version__,
    "workspace_root": str(WORKSPACE_ROOT),
    "input_discovery_method": INPUT_DISCOVERY_METHOD,
}
ENVIRONMENT_PATH = METADATA_DIR / "03_software_environment.json"
ENVIRONMENT_PATH.write_text(json.dumps(environment, indent=2), encoding="utf-8")

no_ml_assertion = {
    "notebook": "03_dependency_graph_and_leakage_compiler.ipynb",
    "no_ml_training_performed": True,
    "no_hyperparameter_tuning_performed": True,
    "no_dft_calculation_performed": True,
}
NO_ML_PATH = METADATA_DIR / "03_no_ml_assertion.json"
NO_ML_PATH.write_text(json.dumps(no_ml_assertion, indent=2), encoding="utf-8")

final_decision_payload = {
    "final_decision": FINAL_DECISION,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "policy_version": POLICY_VERSION,
    "n_target_feature_rows": int(len(compiled)),
    "n_features": int(feature_registry["feature"].nunique()),
    "n_targets": int(len(TARGETS)),
    "class_agreement_rate": float(agreement["class_agreement"].mean()),
    "false_safe_n": int(agreement["false_safe"].sum()),
    "protocol_agreement_rate": float(agreement["all_protocols_agree"].mean()),
    "protocol_count_rows_all_agree": bool(count_audit["all_count_fields_agree"].all()),
    "graph_nodes": int(G.number_of_nodes()),
    "graph_edges": int(G.number_of_edges()),
    "next_notebook": "05_physics_constrained_multitask_electrode_learning.ipynb",
}
FINAL_DECISION_OUTPUT_PATH = METADATA_DIR / "03_final_decision.json"
FINAL_DECISION_OUTPUT_PATH.write_text(json.dumps(final_decision_payload, indent=2), encoding="utf-8")

ready_manifest = {
    "notebook": "03_dependency_graph_and_leakage_compiler.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "policy_version": POLICY_VERSION,
    "input_notebook_02_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "authoritative_compiled_matrix": str(COMPILED_MATRIX_PATH),
    "compiled_feature_sets": str(COMPILED_FEATURE_SETS_PATH),
    "dependency_graph_json": str(GRAPH_JSON_PATH),
    "dependency_graph_graphml": str(GRAPHML_PATH),
    "leakage_rules_yaml": str(RULES_YAML_PATH),
    "compiler_module": str(PACKAGE_DIR / "leakage_compiler.py"),
    "compiler_validation_summary": str(VALIDATION_SUMMARY_PATH),
    "final_decision": FINAL_DECISION,
}
READY_MANIFEST_PATH = METADATA_DIR / "03_compiler_ready_manifest.json"
READY_MANIFEST_PATH.write_text(json.dumps(ready_manifest, indent=2), encoding="utf-8")

event_log = pd.DataFrame([
    {"timestamp_utc": RUN_TIMESTAMP_UTC, "event": "notebook_started", "status": "INFO"},
    {"timestamp_utc": RUN_TIMESTAMP_UTC, "event": "notebook_02_inputs_validated", "status": "PASS"},
    {"timestamp_utc": RUN_TIMESTAMP_UTC, "event": "compiler_rules_executed", "status": "PASS"},
    {"timestamp_utc": RUN_TIMESTAMP_UTC, "event": "expert_compiler_agreement_validated", "status": "PASS"},
    {"timestamp_utc": RUN_TIMESTAMP_UTC, "event": "dependency_graph_written", "status": "PASS"},
    {"timestamp_utc": RUN_TIMESTAMP_UTC, "event": FINAL_DECISION, "status": "PASS"},
])
EVENT_LOG_PATH = LOG_DIR / "03_event_log.csv"
event_log.to_csv(EVENT_LOG_PATH, index=False)

# Remove interpreter cache artifacts before constructing the release-facing manifest.
for cache_dir in OUTPUT_ROOT.rglob("__pycache__"):
    if cache_dir.is_dir():
        shutil.rmtree(cache_dir)
for pyc_file in OUTPUT_ROOT.rglob("*.pyc"):
    pyc_file.unlink(missing_ok=True)

# Build output manifest last, excluding itself until after collection.
output_files = sorted([p for p in OUTPUT_ROOT.rglob("*") if p.is_file()])
manifest_rows = []
for path in output_files:
    manifest_rows.append({
        "relative_path": str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/"),
        "file_name": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })
output_manifest = pd.DataFrame(manifest_rows)
OUTPUT_MANIFEST_PATH = METADATA_DIR / "03_output_file_manifest.csv"
output_manifest.to_csv(OUTPUT_MANIFEST_PATH, index=False)

print("\nNotebook 09B completed successfully.")
print("Final decision:", FINAL_DECISION)
print("Output directory:", OUTPUT_ROOT)
print("Output files recorded:", len(output_manifest))


## Expected successful completion

The final code cell must print:

```text
Final decision: FULL_GO_TO_NOTEBOOK_10B
```

The most important validation conditions are:

- `class_agreement_rate = 1.0`
- `false_safe_n = 0`
- `protocol_agreement_rate = 1.0`
- all 45 protocol-count rows reconstructed exactly

Do not continue to Notebook 05 if the notebook prints a hold decision or raises an exception.
